# Hyperparameter Tuning — Random Search

[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmad-fathan/Deep-Learning-Course/blob/main/hyperparameter-tuning/hyperparameter_tuning.ipynb)

Contoh paling sederhana tuning hyperparameter MLP di TensorFlow/Keras: **Random Search** pada dataset `digits`.

Yang dituning hanya tiga hal: `lr` (skala logaritmik), `units`, `dropout`. `batch_size` tetap 32 dan jumlah epoch tidak dicari karena dipasang **early stopping**.


---
## 1. Impor

Di Colab, TensorFlow dan scikit-learn sudah tersedia, jadi tidak perlu install apa pun.


In [1]:
import random

import numpy as np
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
tf.random.set_seed(SEED)

---
## 2. Data: train / validation / test

Validation dipakai untuk **memilih** konfigurasi. Test hanya dipakai **sekali** di akhir.


In [2]:
digits = load_digits()
x, y = digits.data.astype("float32"), digits.target.astype("int32")

x_tr, x_tmp, y_tr, y_tmp = train_test_split(x, y, test_size=0.4, random_state=SEED, stratify=y)
x_va, x_te, y_va, y_te = train_test_split(x_tmp, y_tmp, test_size=0.5, random_state=SEED, stratify=y_tmp)

print(x_tr.shape, x_va.shape, x_te.shape)

(1078, 64) (359, 64) (360, 64)


---
## 3. Satu kali training

Fungsi `train()` melatih satu model dengan konfigurasi tertentu dan mengembalikan **akurasi validasi terbaik**.
Early stopping dengan `patience=5` membuat jumlah epoch tidak perlu ikut dituning.


In [3]:
def train(lr, units, dropout):
    model = tf.keras.Sequential([
        tf.keras.layers.Input((64,)),
        tf.keras.layers.Dense(units, activation="relu"),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(units // 2, activation="relu"),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    hist = model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=40, batch_size=32,
                     verbose=0, callbacks=[tf.keras.callbacks.EarlyStopping(
                         monitor="val_accuracy", patience=5, restore_best_weights=True)])
    return max(hist.history["val_accuracy"]), model

---
## 4. Random Search

12 percobaan. `lr` diambil pada skala logaritmik (`10 ** U(-4, -1.5)`) supaya nilai kecil ikut tercoba.


In [4]:
best_acc, best_cfg, best_model = 0.0, None, None

for trial in range(1, 13):
    cfg = {"lr": 10 ** random.uniform(-4, -1.5),
           "units": random.choice([32, 64, 128, 256]),
           "dropout": random.uniform(0.0, 0.5)}
    acc, model = train(**cfg)
    print(f"trial {trial:2d}  val_acc={acc:.3f}  lr={cfg['lr']:.4f} "
          f"units={cfg['units']} dropout={cfg['dropout']:.2f}")
    if acc > best_acc:
        best_acc, best_cfg, best_model = acc, cfg, model

print("\nbest:", {k: round(v, 5) for k, v in best_cfg.items()}, f"val_acc={best_acc:.3f}")

trial  1  val_acc=0.925  lr=0.0040 units=32 dropout=0.37


trial  2  val_acc=0.950  lr=0.0049 units=32 dropout=0.30


trial  3  val_acc=0.947  lr=0.0018 units=32 dropout=0.28


trial  4  val_acc=0.958  lr=0.0004 units=128 dropout=0.40


trial  5  val_acc=0.983  lr=0.0011 units=128 dropout=0.08


trial  6  val_acc=0.939  lr=0.0002 units=128 dropout=0.30


... (2 outputs left out)

trial  9  val_acc=0.100  lr=0.0290 units=64 dropout=0.43


trial 10  val_acc=0.975  lr=0.0008 units=128 dropout=0.18


trial 11  val_acc=0.833  lr=0.0002 units=64 dropout=0.27


trial 12  val_acc=0.423  lr=0.0298 units=64 dropout=0.34

best: {'lr': 0.00114, 'units': 128, 'dropout': 0.07774} val_acc=0.983


---
## 5. Uji sekali pada test set

Model terbaik diuji pada test set yang belum pernah dipakai saat memilih konfigurasi.


In [5]:
test_loss, test_acc = best_model.evaluate(x_te, y_te, verbose=0)
print(f"test_acc: {test_acc:.3f}")

test_acc: 0.975


---
## Latihan

1. Tambahkan `batch_size` sebagai hyperparameter dan bandingkan hasilnya.
2. Ganti Random Search dengan Grid Search untuk `lr` dan `units` saja. Berapa kombinasi yang harus dilatih?
3. Pasang `optuna.pruners.HyperbandPruner()` seperti pada Bagian 6 modul.


In [6]:
# TODO: tulis jawaban Latihan di sini